# Domain 1: Agentic Architecture & Orchestration
## Claude Certified Architect – Foundations Certification
**Weight: 27% of scored content** (Heaviest domain)

This notebook provides a comprehensive deep-dive into each task statement in Domain 1.
We cover the theory, knowledge requirements, practical skills, and code examples.

---

## Task 1.1: Design and Implement Agentic Loops for Autonomous Task Execution

### Key Knowledge Areas

**The Agentic Loop Lifecycle:**
1. Send a request to Claude with available tools
2. Inspect `stop_reason` in the response:
   - `"tool_use"` → Claude wants to call a tool → execute it, append result, loop
   - `"end_turn"` → Claude is done → present final response to user
3. Tool results are appended to conversation history so the model reasons about next actions

**Model-Driven Decision Making:**
- Claude *reasons* about which tool to call next based on context
- This is NOT a pre-configured decision tree or fixed tool sequence
- The model dynamically decides based on accumulated context

### Anti-Patterns to Avoid ⚠️
| Anti-Pattern | Why It's Wrong |
|---|---|
| Parsing natural language to determine loop termination | Fragile, model output format may change |
| Setting arbitrary iteration caps as primary stopping mechanism | Masks real issues, may cut off valid work |
| Checking for assistant text content as completion indicator | Unreliable — model may include text alongside tool calls |

In [ ]:
# === AGENTIC LOOP: Core Pattern ===
# This is the fundamental pattern tested on the exam

import anthropic
import json

client = anthropic.Anthropic()  # Requires ANTHROPIC_API_KEY env var

# Define tools available to Claude
tools = [
    {
        "name": "get_weather",
        "description": "Get current weather for a location. Use when the user asks about weather conditions.",
        "input_schema": {
            "type": "object",
            "properties": {
                "location": {"type": "string", "description": "City and state/country"}
            },
            "required": ["location"]
        }
    }
]

def execute_tool(name, input_data):
    """Simulate tool execution — in production, these call real backends."""
    if name == "get_weather":
        return {"temp": "22°C", "condition": "Sunny", "location": input_data["location"]}
    return {"error": f"Unknown tool: {name}"}

def run_agentic_loop(user_message, tools, max_iterations=10):
    """
    Core agentic loop pattern.
    Key exam point: Loop continues when stop_reason == 'tool_use'
                    Loop terminates when stop_reason == 'end_turn'
    """
    messages = [{"role": "user", "content": user_message}]
    
    for iteration in range(max_iterations):  # Safety cap (NOT primary stop mechanism)
        response = client.messages.create(
            model="claude-sonnet-4-20250514",
            max_tokens=1024,
            tools=tools,
            messages=messages
        )
        
        # ✅ CORRECT: Check stop_reason to determine loop continuation
        if response.stop_reason == "end_turn":
            # Claude is done — extract and return the final text
            return [b.text for b in response.content if b.type == "text"]
        
        elif response.stop_reason == "tool_use":
            # Claude wants to use tools — execute them and continue
            # Append assistant's response (includes tool_use blocks)
            messages.append({"role": "assistant", "content": response.content})
            
            # Execute each tool call and collect results
            tool_results = []
            for block in response.content:
                if block.type == "tool_use":
                    result = execute_tool(block.name, block.input)
                    tool_results.append({
                        "type": "tool_result",
                        "tool_use_id": block.id,
                        "content": json.dumps(result)
                    })
            
            # Append tool results so Claude can reason about them
            messages.append({"role": "user", "content": tool_results})
    
    return ["Max iterations reached"]  # Safety fallback

# Example usage (uncomment to run with a real API key):
# result = run_agentic_loop("What's the weather in London?", tools)
# print(result)
print("Agentic loop pattern defined. Set ANTHROPIC_API_KEY to run.")

### Exam Tip 💡

> **Question pattern:** "Your agent terminates prematurely / runs forever. What's wrong?"
>
> Always look for: Is the code checking `stop_reason` correctly?
> The correct pattern is:
> - `stop_reason == "tool_use"` → continue looping
> - `stop_reason == "end_turn"` → stop and return response

---

## Task 1.2: Orchestrate Multi-Agent Systems with Coordinator-Subagent Patterns

### Hub-and-Spoke Architecture

The coordinator pattern is the **central architecture** for multi-agent systems:

```
                    ┌─────────────┐
                    │ Coordinator │
                    │   Agent     │
                    └──────┬──────┘
              ┌────────────┼────────────┐
              ▼            ▼            ▼
        ┌──────────┐ ┌──────────┐ ┌──────────┐
        │ Search   │ │ Analysis │ │ Synthesis│
        │ Subagent │ │ Subagent │ │ Subagent │
        └──────────┘ └──────────┘ └──────────┘
```

### Critical Knowledge Points

1. **Isolated Context**: Subagents do NOT inherit the coordinator's conversation history automatically
2. **Coordinator Responsibilities**:
   - Task decomposition
   - Delegation to appropriate subagents
   - Result aggregation
   - Error handling
   - Deciding which subagents to invoke based on query complexity
3. **Common Pitfall**: Overly narrow task decomposition by the coordinator → incomplete coverage
   - Example: Decomposing "AI in creative industries" into only visual arts subtasks, missing music/writing/film

### Key Skills

- **Dynamic subagent selection**: Don't always route through the full pipeline
- **Minimize duplication**: Assign distinct subtopics or source types to each agent
- **Iterative refinement**: Coordinator evaluates synthesis, re-delegates if gaps found
- **Observability**: Route ALL subagent communication through the coordinator

In [ ]:
# === MULTI-AGENT COORDINATOR PATTERN ===

from dataclasses import dataclass
from typing import Optional

@dataclass
class SubagentResult:
    agent_name: str
    findings: list
    metadata: dict
    success: bool
    error: Optional[str] = None

class CoordinatorAgent:
    """Hub-and-spoke coordinator pattern.
    
    Exam-relevant points:
    - Coordinator manages ALL inter-subagent communication
    - Subagents receive explicit context (not inherited)
    - Dynamic subagent selection based on query analysis
    """
    
    def __init__(self):
        self.subagents = {}
        self.results_history = []
    
    def register_subagent(self, name, description, tools):
        self.subagents[name] = {"description": description, "tools": tools}
    
    def analyze_and_decompose(self, query):
        """
        CRITICAL: The coordinator must decompose broadly.
        
        Anti-pattern: Decomposing 'AI in creative industries' into only:
        - 'AI in digital art'
        - 'AI in graphic design' 
        - 'AI in photography'
        This misses music, writing, film, etc.
        """
        # In production, Claude analyzes the query and determines:
        # 1. Which subagents to invoke
        # 2. What subtasks to create
        # 3. Whether parallel or sequential execution is needed
        return {
            "subtasks": ["subtask_1", "subtask_2"],
            "required_agents": ["search", "analysis"],
            "parallel_ok": True
        }
    
    def delegate_to_subagent(self, agent_name, task, context):
        """
        Key exam point: Context must be EXPLICITLY passed.
        Subagents don't inherit coordinator's conversation history.
        """
        # Include complete findings from prior agents in the prompt
        prompt = f"""Task: {task}
        
Prior findings from other agents:
{context}

Provide structured output with source attribution."""
        # In production: spawn subagent via Task tool
        return SubagentResult(
            agent_name=agent_name,
            findings=["finding_1", "finding_2"],
            metadata={"source": "example"},
            success=True
        )
    
    def evaluate_coverage(self, synthesis_result, original_query):
        """Iterative refinement: check for gaps and re-delegate if needed."""
        # Coordinator evaluates if synthesis covers all aspects
        # If gaps found: re-delegate to search/analysis with targeted queries
        pass

# Demo
coordinator = CoordinatorAgent()
coordinator.register_subagent("search", "Web search agent", ["web_search"])
coordinator.register_subagent("analysis", "Document analysis agent", ["analyze_doc"])
coordinator.register_subagent("synthesis", "Synthesis agent", ["synthesize"])
print(f"Coordinator registered {len(coordinator.subagents)} subagents: {list(coordinator.subagents.keys())}")

---

## Task 1.3: Configure Subagent Invocation, Context Passing, and Spawning

### The Task Tool

- The **Task tool** is the mechanism for spawning subagents
- `allowedTools` must include `"Task"` for a coordinator to invoke subagents
- Subagent context must be **explicitly provided** in the prompt

### AgentDefinition Configuration

```python
agent_definition = {
    "name": "search_agent",
    "description": "Searches web for relevant information",
    "system_prompt": "You are a research assistant...",
    "allowedTools": ["web_search", "fetch_url"],  # Scoped tools
}
```

### Context Passing Best Practices

| Practice | Description |
|---|---|
| Include complete findings | Pass web search results AND document analysis to synthesis agent |
| Structured data formats | Separate content from metadata (source URLs, page numbers) |
| Preserve attribution | Include source URLs, document names alongside findings |

### Parallel Subagent Execution

- Spawn parallel subagents by emitting **multiple Task tool calls in a single coordinator response**
- NOT across separate turns

### Fork-Based Session Management

- `fork_session` creates independent branches from a shared analysis baseline
- Useful for exploring divergent approaches (e.g., comparing two testing strategies)

In [ ]:
# === SUBAGENT CONTEXT PASSING PATTERN ===

def build_subagent_prompt(task, prior_findings, metadata):
    """
    Exam-critical: Subagents receive EXPLICIT context.
    They do NOT automatically inherit parent context.
    
    Best practice: Use structured formats to separate 
    content from metadata for attribution preservation.
    """
    
    # Structure findings with metadata for attribution
    findings_block = ""
    for finding in prior_findings:
        findings_block += f"""
--- Finding ---
Source: {finding['source_url']}
Document: {finding['document_name']}
Page: {finding.get('page_number', 'N/A')}
Content: {finding['content']}
---
"""
    
    prompt = f"""## Task
{task}

## Prior Findings from Other Agents
{findings_block}

## Research Goals
- Synthesize findings into a coherent narrative
- Preserve source attribution for all claims
- Flag any gaps or conflicting information

## Quality Criteria
- Every claim must include its source
- Distinguish well-supported findings from contested ones
- Include publication dates for temporal context
"""
    return prompt

# Example: Building context for synthesis subagent
prior = [
    {
        "source_url": "https://example.com/ai-music",
        "document_name": "AI in Music Production 2024",
        "page_number": 12,
        "content": "AI composition tools grew 340% in adoption during 2024"
    },
    {
        "source_url": "https://example.com/ai-film",
        "document_name": "Film Industry AI Report",
        "content": "73% of VFX studios now use AI-assisted tools"
    }
]

prompt = build_subagent_prompt(
    task="Synthesize findings on AI impact across creative industries",
    prior_findings=prior,
    metadata={"topic": "AI in creative industries"}
)
print(prompt[:500])

---

## Task 1.4: Implement Multi-Step Workflows with Enforcement and Handoff Patterns

### Programmatic vs Prompt-Based Enforcement

This is a **critical exam distinction**:

| Approach | Guarantee Level | Use When |
|---|---|---|
| **Programmatic enforcement** (hooks, prerequisite gates) | Deterministic ✅ | Financial operations, identity verification, compliance |
| **Prompt-based guidance** | Probabilistic ⚠️ | Non-critical ordering preferences |

> **Key Exam Point:** When deterministic compliance is required (e.g., identity verification
> before financial operations), prompt instructions alone have a **non-zero failure rate**.

### Structured Handoff Protocols

When escalating to human agents, compile:
- Customer ID
- Root cause analysis
- Refund amount
- Recommended action

Human agents may **not** have access to the conversation transcript!

In [ ]:
# === PROGRAMMATIC PREREQUISITE ENFORCEMENT ===

class ToolPrerequisiteEnforcer:
    """
    Blocks downstream tool calls until prerequisites are met.
    
    Exam scenario: Block process_refund until get_customer 
    has returned a verified customer ID.
    """
    
    def __init__(self):
        self.completed_steps = set()
        self.prerequisites = {
            # tool_name: [required prerequisite tools]
            "lookup_order": ["get_customer"],
            "process_refund": ["get_customer", "lookup_order"],
            "escalate_to_human": []  # Can always escalate
        }
    
    def can_execute(self, tool_name):
        """Check if all prerequisites are met."""
        required = self.prerequisites.get(tool_name, [])
        missing = [r for r in required if r not in self.completed_steps]
        return len(missing) == 0, missing
    
    def intercept_tool_call(self, tool_name, tool_input):
        """
        Called BEFORE executing any tool.
        Returns (allowed, message).
        """
        allowed, missing = self.can_execute(tool_name)
        if not allowed:
            return False, f"Blocked: {tool_name} requires {missing} to complete first"
        return True, "Proceed"
    
    def mark_completed(self, tool_name):
        self.completed_steps.add(tool_name)

# Demo
enforcer = ToolPrerequisiteEnforcer()

# Try to call process_refund before verification
allowed, msg = enforcer.intercept_tool_call("process_refund", {})
print(f"process_refund without verification: allowed={allowed}, msg='{msg}'")

# Complete prerequisites
enforcer.mark_completed("get_customer")
enforcer.mark_completed("lookup_order")

# Now try again
allowed, msg = enforcer.intercept_tool_call("process_refund", {})
print(f"process_refund after verification: allowed={allowed}, msg='{msg}'")

### Structured Handoff Example

When the agent cannot resolve an issue or the customer requests a human:

In [ ]:
# === STRUCTURED HANDOFF SUMMARY ===

def compile_handoff_summary(session_data):
    """
    Human agents may NOT have access to the conversation transcript.
    The handoff must be self-contained.
    """
    return {
        "customer_id": session_data.get("customer_id"),
        "customer_name": session_data.get("customer_name"),
        "root_cause": session_data.get("root_cause"),
        "issue_summary": session_data.get("issue_summary"),
        "amount_involved": session_data.get("refund_amount"),
        "recommended_action": session_data.get("recommendation"),
        "steps_already_taken": session_data.get("steps_taken", []),
        "escalation_reason": session_data.get("escalation_reason")
    }

# Example
handoff = compile_handoff_summary({
    "customer_id": "CUST-12345",
    "customer_name": "Jane Smith",
    "root_cause": "Defective product received — photos confirm damage",
    "issue_summary": "Customer received damaged laptop, requests full refund of $1,299",
    "refund_amount": 1299.00,
    "recommendation": "Approve full refund — clear product defect with photo evidence",
    "steps_taken": ["Verified customer identity", "Confirmed order #ORD-98765", "Reviewed damage photos"],
    "escalation_reason": "Refund amount exceeds $500 automated threshold"
})

for k, v in handoff.items():
    print(f"{k}: {v}")

---

## Task 1.5: Apply Agent SDK Hooks for Tool Call Interception and Data Normalization

### Hook Patterns

| Hook Type | Purpose | Example |
|---|---|---|
| **PostToolUse** | Transform tool results before model processes them | Normalize timestamps from Unix → ISO 8601 |
| **Tool Call Interception** | Block/redirect outgoing tool calls | Block refunds > $500 |

### Hooks vs Prompts

- **Hooks** = Deterministic guarantees (always enforced)
- **Prompts** = Probabilistic compliance (usually followed, but not guaranteed)

> **Rule of thumb:** Use hooks when business rules require **guaranteed** compliance.

In [ ]:
# === AGENT SDK HOOKS ===

class PostToolUseHook:
    """Normalize heterogeneous data formats from different MCP tools."""
    
    @staticmethod
    def normalize_timestamp(value):
        """Convert various timestamp formats to ISO 8601."""
        from datetime import datetime
        
        if isinstance(value, (int, float)):  # Unix timestamp
            return datetime.fromtimestamp(value).isoformat()
        elif isinstance(value, str):
            # Already ISO format — return as-is
            return value
        return str(value)
    
    @staticmethod
    def normalize_status_code(code):
        """Convert numeric status codes to human-readable strings."""
        status_map = {0: "pending", 1: "processing", 2: "completed", 3: "failed"}
        return status_map.get(code, f"unknown_{code}")
    
    def process(self, tool_name, tool_result):
        """Transform tool results before the model sees them."""
        if "timestamp" in tool_result:
            tool_result["timestamp"] = self.normalize_timestamp(tool_result["timestamp"])
        if "status" in tool_result and isinstance(tool_result["status"], int):
            tool_result["status"] = self.normalize_status_code(tool_result["status"])
        return tool_result


class ToolCallInterceptor:
    """Block policy-violating tool calls and redirect to alternatives."""
    
    def __init__(self, refund_limit=500):
        self.refund_limit = refund_limit
    
    def intercept(self, tool_name, tool_input):
        """
        Returns (proceed, alternative_action)
        """
        if tool_name == "process_refund":
            amount = tool_input.get("amount", 0)
            if amount > self.refund_limit:
                return False, {
                    "action": "escalate_to_human",
                    "reason": f"Refund ${amount} exceeds ${self.refund_limit} limit",
                    "original_request": tool_input
                }
        return True, None

# Demo
hook = PostToolUseHook()
raw_result = {"timestamp": 1709312400, "status": 2, "order_id": "ORD-123"}
normalized = hook.process("lookup_order", raw_result.copy())
print(f"Before: {raw_result}")
print(f"After:  {normalized}")

interceptor = ToolCallInterceptor(refund_limit=500)
proceed, alt = interceptor.intercept("process_refund", {"amount": 750})
print(f"\n$750 refund: proceed={proceed}, redirect to: {alt['action']}")

---

## Task 1.6: Design Task Decomposition Strategies for Complex Workflows

### Two Patterns

| Pattern | When to Use | Example |
|---|---|---|
| **Prompt Chaining** (fixed sequential) | Predictable multi-aspect reviews | Per-file analysis → cross-file integration pass |
| **Dynamic Adaptive Decomposition** | Open-ended investigation tasks | Discover structure → identify high-impact areas → prioritize |

### Code Review Decomposition

For large PRs (e.g., 14 files):
1. **Per-file local analysis** — analyze each file individually
2. **Cross-file integration pass** — examine data flow across files

This prevents **attention dilution** (inconsistent depth, contradictory feedback).

### Open-Ended Task Decomposition

For tasks like "add comprehensive tests to a legacy codebase":
1. First map the codebase structure
2. Identify high-impact areas
3. Create a prioritized plan
4. Adapt as dependencies are discovered

In [ ]:
# === TASK DECOMPOSITION PATTERNS ===

class PromptChainingDecomposer:
    """Fixed sequential pipeline for predictable workflows."""
    
    def decompose_code_review(self, files):
        """Split large PR review to avoid attention dilution."""
        tasks = []
        
        # Phase 1: Per-file local analysis
        for f in files:
            tasks.append({
                "phase": "local_analysis",
                "file": f,
                "instruction": f"Analyze {f} for bugs, security issues, and local patterns"
            })
        
        # Phase 2: Cross-file integration pass
        tasks.append({
            "phase": "integration_analysis",
            "files": files,
            "instruction": "Examine cross-file data flow, API consistency, and integration issues"
        })
        
        return tasks


class AdaptiveDecomposer:
    """Dynamic decomposition based on intermediate findings."""
    
    def decompose_legacy_testing(self, codebase_info):
        """
        Open-ended task: Generate subtasks adaptively.
        Each step's output informs the next step's tasks.
        """
        return [
            {"step": 1, "action": "Map codebase structure and module boundaries"},
            {"step": 2, "action": "Identify high-impact areas (most bugs, most changes, least coverage)"},
            {"step": 3, "action": "Create prioritized test plan based on findings"},
            {"step": 4, "action": "Generate tests, adapting as dependencies are discovered"}
        ]

# Demo: Code review decomposition
decomposer = PromptChainingDecomposer()
pr_files = ["auth.py", "models.py", "api/routes.py", "tests/test_auth.py"]
tasks = decomposer.decompose_code_review(pr_files)
print(f"Decomposed PR review into {len(tasks)} tasks:")
for t in tasks:
    print(f"  [{t['phase']}] {t.get('file', 'all files')}")

---

## Task 1.7: Manage Session State, Resumption, and Forking

### Session Management Commands

| Command | Purpose |
|---|---|
| `--resume <session-name>` | Continue a specific prior conversation |
| `fork_session` | Create independent branches from shared baseline |
| Start fresh with summary | More reliable when prior tool results are stale |

### Decision Guide

```
Has the code changed since last session?
├── No → Resume session (prior context still valid)
├── Partially → Resume + inform about specific file changes
└── Significantly → Start fresh with injected summary
```

### Fork Session Use Cases

- Comparing two testing strategies from a shared codebase analysis
- Exploring alternative refactoring approaches
- Running parallel experiments from a common baseline

### Key Exam Points

- When resuming after code modifications, **inform the agent** about changes to previously analyzed files
- Starting fresh with a structured summary is **more reliable** than resuming with stale tool results
- Use `/compact` to reduce context usage during extended exploration sessions

---

## Domain 1 Summary & Exam Preparation

### Key Concepts to Remember

1. **Agentic Loop**: `stop_reason` is the sole loop control mechanism
2. **Coordinator Pattern**: Hub-and-spoke, isolated context, explicit passing
3. **Task Tool**: `allowedTools` must include `"Task"` for subagent spawning
4. **Enforcement**: Programmatic > Prompt-based for critical business rules
5. **Hooks**: PostToolUse for normalization, interceptors for policy enforcement
6. **Decomposition**: Prompt chaining (predictable) vs adaptive (open-ended)
7. **Sessions**: Resume (context valid) vs fresh start (context stale)

### Sample Exam Question Pattern

> Your agent skips `get_customer` in 12% of cases and calls `lookup_order` directly.
> What's the fix?
>
> ✅ **Programmatic prerequisite** that blocks `lookup_order` until `get_customer` completes
>
> ❌ Enhanced system prompt (probabilistic, not deterministic)
>
> ❌ Few-shot examples (still probabilistic)
>
> ❌ Routing classifier (addresses availability, not ordering)